In [1]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Safe to run locally: it does nothing outside Colab.
try:
    import google.colab  # noqa: F401
    %pip install -q git+https://github.com/C0PEP0D/SoftMobility.git
except ImportError:
    pass

# Tutorial 13 — Rotating elastic filament (Coq 2009 / Delmotte 2015 fig 13)

A flexible filament is anchored at one end on a rotation axis and
the anchor is forced to rotate at angular rate $\zeta$ around that
axis with a fixed tilt $\psi$ relative to it. Viscous drag drives
the chain into a steady, body-frame chiral shape that wraps onto the
rotation axis as the dimensionless rotation rate $\mathrm{Sp}^4$
increases. We reproduce this canonical setup of Coq, du Roure,
Marthelot, Bartolo & Fermigier (2008, 2009) — also Fig. 13 of
Delmotte, Climent & Plouraboué (2015) — at three values
$\mathrm{Sp}^4 = 1, 40, 500$ corresponding to the *stiff*,
*medium* and *soft* regimes.

**Why this is a clamped-anchor problem.** The body's six-component
lab pose is *prescribed* by the actuator, so the body's
six-component velocity
$\mathbf{v}_0 = [\mathbf{u}_0, \boldsymbol{\omega}_0]$
becomes an input of the soft mobility equation rather than an
output. The six-component reaction force the anchor exerts on
sphere 0, $\mathbf{f}_0 = [\mathbf{F}_0, \mathbf{T}_0]$, is the
corresponding new unknown. The mobility equation is partitioned
(Appendix A 5 of the manuscript): the top six rows form a
$6\times 6$ linear system for $\mathbf{f}_0$ given $\mathbf{v}_0$;
the bottom $N_Q$ rows then yield the deformation rate
$\dot{\mathbf{Q}}$.

The library exposes this via
[`FlowBodyRollout.rollout_clamped_anchor`](../classes/solver.py),
which takes the actuator's lab-frame **position** and **velocity**
as two callables of $t$ plus the body's initial orientation, and
returns the trajectory of the deformation DOFs together with the
reaction $\mathbf{f}_0$ at every step. The body's Rodrigues vector
is integrated forward internally with the same Bortz/RK4 scheme as
the standard rollout, so it stays singularity-free for unbounded
rotations.

In [2]:
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go

import softmobility as sm
from softmobility.tutorials import figstyle

figstyle.apply()
FIGDIR = "figures"

np.set_printoptions(precision=3, suppress=True)

## 1 — Filament geometry and dimensionless control

We model the filament as $N=8$ beads of radius $a=1$ connected by
rigid bonds of length $2a$. Bending modulus $K_b=30$, viscosity
$\mu=1$, per-unit-length transverse drag
$\gamma_\perp \approx 4\pi\mu$ (RFT slender body). The Sperm
number (Delmotte convention) is
$$
\mathrm{Sp}^4 = \frac{L^3\,\zeta\,\gamma_\perp}{K_b},
$$
with $L = (N-1)\cdot 2a$. Tilt angle $\psi = \arcsin(0.259)
\approx 15^\circ$ from the rotation axis (Coq's experimental
value). Geometry and elasticity are held fixed across the three
Sp⁴ runs; only $\zeta$ changes.

In [3]:
N = 8
a = 1.0
mu = 1.0
K_b = 30.0
psi = float(np.arcsin(0.259))                  # ~15° tilt
gamma_perp = 4 * np.pi * mu                    # RFT slender body

L = (N - 1) * 2 * a
tau_el = gamma_perp * L ** 3 / K_b              # elastic relaxation time
print(f"L = {L:.1f}, K_b = {K_b}, tau_el = {tau_el:.1f}")

fiber = sm.FlexibleFiber(
    n_beads=N, radius=a, bending_rigidity=K_b, mass=0.0, planar=False,
)
rollout = sm.FlowBodyRollout(
    soft_body=fiber, flow=sm.no_flow(),
    input_map={"gravity": sm.gravity_field(g=0.0)},
)
print(f"FlexibleFiber: Nspheres = {fiber.Nspheres}, Ndof = {fiber.Ndof}")

L = 14.0, K_b = 30.0, tau_el = 1149.4


FlexibleFiber: Nspheres = 8, Ndof = 14


## 2 — Anchor pose and velocity

Anchor sits on the rotation axis, so the lab-frame translational
velocity is identically zero; the lab-frame angular velocity is
the constant
$\boldsymbol{\omega}_{\rm lab} = \zeta\,\mathbf{e}_x$.
The body's initial orientation tilts $\mathbf{e}_x$ by $\psi$
around $\mathbf{e}_y$ — so as the body rotates around
$\mathbf{e}_x$, the chain (which extends along the body $\mathbf{e}_x$)
precesses on a cone of half-angle $\psi$ around the rotation axis.

`rollout_clamped_anchor` integrates the body's Rodrigues vector
forward via Bortz from the supplied $\boldsymbol{\omega}_{\rm lab}(t)$ —
no singular axis-angle reconstruction is needed.

In [4]:
def anchor_position_fn(t):
    """Anchor sits on the lab x-axis (rotation axis)."""
    return jnp.zeros(3)


def make_anchor_velocity_fn(zeta):
    omega_lab = jnp.array([zeta, 0.0, 0.0])

    def anchor_velocity_fn(t):
        return jnp.concatenate([jnp.zeros(3), omega_lab])

    return anchor_velocity_fn


INIT_ORIENTATION = jnp.array([0.0, psi, 0.0])

## 3 — Run one $\mathrm{Sp}^4$ to show the API

We integrate for $\sim 5\,\tau_{\rm el}$ so the deformation
$\mathbf{Q}$ reaches a body-frame steady state. The intermediate
$\mathrm{Sp}^4 = 40$ regime is the slowest to relax (the slowest
elastic mode dominates); the stiff and soft endpoints settle in
fewer $\tau_{\rm el}$ but using the same horizon for all three
keeps the code uniform.

The integrator step is $\Delta t = 0.2\,(2a)^4\mu/K_b$, four
times the conservative default of the existing `rollout` —
well within RK4 stability for the bending modes.

In [5]:
def integrate_to_steady_state(sp4, n_tau=5.0, dt_safety=0.2):
    zeta = sp4 * K_b / (L ** 3 * gamma_perp)
    dt = dt_safety * (2 * a) ** 4 * mu / K_b
    n_steps = int(round(n_tau * tau_el / dt))
    positions, orientations, dofs, f_0 = rollout.rollout_clamped_anchor(
        dt=dt,
        n_steps=n_steps,
        anchor_position_fn=anchor_position_fn,
        anchor_velocity_fn=make_anchor_velocity_fn(zeta),
        init_orientation=INIT_ORIENTATION,
        init_dofs=jnp.zeros(fiber.Ndof),
    )
    positions.block_until_ready()
    return zeta, dt, n_steps, np.asarray(dofs), np.asarray(f_0)


zeta, dt, n_steps, dofs_traj, f_0_traj = integrate_to_steady_state(40.0)
print(f"Sp**4 = 40:  zeta = {zeta:.3e}, dt = {dt:.4f}, n_steps = {n_steps}")
print(f"final max|Q|       = {np.max(np.abs(dofs_traj[-1])):.3e}")
print(f"final max|f_0|     = {np.max(np.abs(f_0_traj[-1])):.3e}")
Q_dot_end = (dofs_traj[-1] - dofs_traj[-2]) / dt
ratio = np.max(np.abs(Q_dot_end)) * tau_el
print(f"final max|Q_dot|*tau_el = {ratio:.3e}  (steady state if << 1)")

Sp**4 = 40:  zeta = 3.480e-02, dt = 0.1067, n_steps = 53878
final max|Q|       = 3.443e-01
final max|f_0|     = 5.574e+00
final max|Q_dot|*tau_el = 4.335e-03  (steady state if << 1)


## 4 — Lab-frame snapshots over one rotation period

At steady state the chain has a fixed body-frame shape
$\mathbf{Q}_\star$; in the lab frame this shape rotates rigidly
around $\mathbf{e}_x$. The figure superimposes `n_phases`
snapshots of the chain over one rotation period as red lines
(default 16; pass `n_phases=12` etc. to coarsen), with grey
shadow projections on the three back panels of the bounding box
and the rotation axis as a dotted guide. Style matches
`fig_3d_com.pdf` of tutorial 11.

In [6]:
def _Rx(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def _Ry(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def body_frame_positions(dofs):
    """Body-frame bead positions (Nspheres, 3) for the given dofs."""
    out = np.zeros((fiber.Nspheres, 3))
    design_np = np.asarray(fiber.design_defaults)
    t_dummy = np.array([0.0])
    for i in range(fiber.Nspheres):
        out[i] = np.asarray(
            fiber.spheres[i].position(dofs, design_np, t_dummy)
        )
    return out


def make_figure(sp4, dofs_steady, zeta, save_name, n_phases=16):
    body_pos = body_frame_positions(np.asarray(dofs_steady))
    period = 2 * np.pi / float(zeta)
    times = np.linspace(0.0, period, n_phases, endpoint=False)
    R_y_psi = _Ry(psi)

    chains_lab = np.stack(
        [body_pos @ (_Rx(zeta * t) @ R_y_psi).T for t in times], axis=0,
    )
    pad = 0.10 * float(L)
    bounds = (
        (float(chains_lab[..., 0].min()) - pad,
         float(chains_lab[..., 0].max()) + pad),
        (float(chains_lab[..., 1].min()) - pad,
         float(chains_lab[..., 1].max()) + pad),
        (float(chains_lab[..., 2].min()) - pad,
         float(chains_lab[..., 2].max()) + pad),
    )

    fig = figstyle.figure_3d(size="full", aspect=1.0)
    figstyle.add_back_panels(fig, *bounds, color=figstyle.COLORS["black"], width=1.0)

    # Rotation axis (lab e_x) as a thin dotted guide.
    fig.add_trace(go.Scatter3d(
        x=[bounds[0][0], bounds[0][1]], y=[0.0, 0.0], z=[0.0, 0.0],
        mode="lines",
        line=dict(color=figstyle.COLORS["grey"], width=1, dash="dot"),
        showlegend=False, hoverinfo="skip",
    ))

    # Per-phase shadows on the three back panels.
    for chain in chains_lab:
        for plane in ("xy_low", "xz_low", "yz_low"):
            figstyle.add_shadow(
                fig, chain[:, 0], chain[:, 1], chain[:, 2],
                plane=plane, bounds=bounds,
                color=figstyle.COLORS["grey"], width=1.0, opacity=0.4,
            )

    # Red chain centerlines (no markers).
    for chain in chains_lab:
        fig.add_trace(go.Scatter3d(
            x=chain[:, 0], y=chain[:, 1], z=chain[:, 2],
            mode="lines",
            line=dict(color=figstyle.COLORS["red"], width=2),
            opacity=0.7,
            showlegend=False, hoverinfo="skip",
        ))

    fig.update_layout(scene=dict(
        xaxis=dict(title="<i>e</i><sub>1</sub>", range=list(bounds[0])),
        yaxis=dict(title="<i>e</i><sub>2</sub>", range=list(bounds[1])),
        zaxis=dict(title="<i>e</i><sub>3</sub>", range=list(bounds[2])),
    ))
    figstyle.save(fig, save_name, figdir=FIGDIR)
    return fig


fig_medium = make_figure(40.0, dofs_traj[-1], zeta, "fig_rotating_medium")
fig_medium.show()

## 5 — $\mathrm{Sp}^4$ scan: stiff, medium, soft

Repeating the rollout for $\mathrm{Sp}^4 \in \{1, 40, 500\}$
produces three figures showing the chain shape sweep:

* **Stiff** ($\mathrm{Sp}^4 = 1$): chain essentially straight,
  tilted by $\psi$, traces a clean cone.
* **Medium** ($\mathrm{Sp}^4 = 40$): visible bending; the swept
  volume is a leaf.
* **Soft** ($\mathrm{Sp}^4 = 500$): the linearised bending of
  `FlexibleFiber` (valid for joint angles ≲ π/4) saturates around
  $\max|\mathbf{Q}| \approx 0.35$. The shape between
  $\mathrm{Sp}^4 = 40$ and $\mathrm{Sp}^4 = 500$ doesn't differ as
  much as Coq's Fig. 4 would predict — that's a *model* limit, not
  a partitioning bug. The reaction torque $|\mathbf{f}_0|$ still
  scales correctly with $\zeta$, growing by an order of magnitude
  between 40 and 500.

Each figure is saved as
`figures/fig_rotating_{stiff|medium|soft}.pdf`.

In [7]:
labels = {1.0: "stiff", 40.0: "medium", 500.0: "soft"}
for sp4 in (1.0, 40.0, 500.0):
    zeta, dt, n_steps, dofs_traj, f_0_traj = integrate_to_steady_state(sp4)
    Q_dot_end = (dofs_traj[-1] - dofs_traj[-2]) / dt
    print(
        f"Sp**4 = {sp4:>5g}  zeta = {zeta:.3e}  "
        f"max|Q| = {np.max(np.abs(dofs_traj[-1])):.3e}  "
        f"max|f_0| = {np.max(np.abs(f_0_traj[-1])):.3e}  "
        f"max|Q_dot|*tau_el = {np.max(np.abs(Q_dot_end))*tau_el:.3e}"
    )
    fig = make_figure(sp4, dofs_traj[-1], zeta,
                       f"fig_rotating_{labels[sp4]}")
    fig.show()

Sp**4 =     1  zeta = 8.700e-04  max|Q| = 1.826e-01  max|f_0| = 9.250e-01  max|Q_dot|*tau_el = 1.606e-04


Sp**4 =    40  zeta = 3.480e-02  max|Q| = 3.443e-01  max|f_0| = 5.574e+00  max|Q_dot|*tau_el = 4.335e-03


Sp**4 =   500  zeta = 4.350e-01  max|Q| = 3.462e-01  max|f_0| = 6.975e+01  max|Q_dot|*tau_el = 1.204e-04


## Notes

* **Anchor reaction force.** $\mathbf{f}_0$ is the force the
  external actuator exerts on sphere 0 to enforce the prescribed
  pose. Its component along the rotation axis is the propulsive
  thrust the filament generates against the actuator (Coq Fig. 5);
  here it grows monotonically with $\zeta$ as expected.
* **Linearised bending.** `FlexibleFiber` uses a small-angle
  expansion of the joint-angle bending energy, accurate for joint
  angles below ~π/4. The $\max|\mathbf{Q}|$ saturation at
  $\mathrm{Sp}^4 \gtrsim 40$ in the scan above is the practical
  reach of this model.
* **Frame conventions.** $\mathbf{v}_0$ and the input pose are
  supplied in the lab frame; $\mathbf{f}_0$ comes back in the
  body frame; the internal mobility tensors are body-frame too.
  The conversions happen inside `rollout_clamped_anchor`.
  Math: Appendix A 5 of Article 3.
* **Differentiability.** `rollout_clamped_anchor` is pure-functional
  and JAX-compatible, so the steady-state shape and propulsive
  thrust are differentiable with respect to design parameters
  (`bending_rigidity`, `radius`) and to the actuation
  (any closure parameter inside `make_anchor_velocity_fn`).